# 05: Agent Execution with Vertex AI Search

This notebook shows the **integrated agent experience** with `vertex_search` wired in
alongside the five standard tools. Notebook 04 demonstrated `vertex_search` in isolation;
here we see how the agent plans, decides which tool to reach for, and how the execution
trace differs between knowledge-base questions and public-web questions.

## What You'll Learn

1. How `vertex_search` is conditionally added to the agent's tool list
2. Inspecting which tools the agent currently has available
3. Running a **knowledge base question** — when the agent should pick `vertex_search`
4. Running a **public web question** — when the agent should pick `google_search`
5. Reading the plan and tool call trace to verify tool selection
6. Side-by-side comparison: KB answer vs web answer

## Prerequisites

Complete Notebooks 01–04. You'll need:
- `GOOGLE_API_KEY` (agent and Google Search)
- `VERTEX_AI_DATASTORE_ID` (Vertex AI Search data store — used in Section 3)
- `GOOGLE_CLOUD_LOCATION` (default: `us-central1`)
- ADC authentication — automatic on GCE/Coder workspaces

In [ ]:
import os
from pathlib import Path

from aieng.agent_evals.configs import Configs
from aieng.agent_evals.knowledge_qa import KnowledgeGroundedAgent
from aieng.agent_evals.knowledge_qa.notebook import display_response, run_with_display
from dotenv import load_dotenv
from rich.console import Console
from rich.panel import Panel
from rich.table import Table


if Path("").absolute().name == "eval-agents":
    print(f"Working directory: {Path('').absolute()}")
else:
    os.chdir(Path("").absolute().parent.parent)
    print(f"Working directory set to: {Path('').absolute()}")

load_dotenv(verbose=True)
console = Console(width=100)
config = Configs()  # type: ignore[call-arg]

## 1. Inspecting the Agent's Tool List

`vertex_search` is added **conditionally** — only when `VERTEX_AI_DATASTORE_ID` is set.
When the environment variable is absent the agent falls back to its five standard tools.

This cell shows exactly which tools the agent has in the current environment.

In [ ]:
agent = KnowledgeGroundedAgent(enable_planning=True)

# The ADK Agent exposes its tools on ._agent.tools
tool_names = [t.func.__name__ if hasattr(t, "func") else str(t) for t in agent._agent.tools]

t = Table(title="Agent Tool List")
t.add_column("#", style="dim", justify="right", width=3)
t.add_column("Tool", style="cyan")
t.add_column("Source", style="dim")
for i, name in enumerate(tool_names, 1):
    source = "[yellow]Vertex AI Search (private KB)[/yellow]" if name == "vertex_search" else "Standard"
    t.add_row(str(i), name, source)
console.print(t)

if config.vertex_datastore_id:
    console.print(f"[green]✓[/green] vertex_search loaded — data store: {config.vertex_datastore_id[:60]}")
else:
    console.print(
        "[yellow]⚠[/yellow] VERTEX_AI_DATASTORE_ID not set. "
        "vertex_search will not appear in the tool list.\n"
        "Set it in .env and restart the kernel to enable it."
    )

## 2. System Prompt — Tool Descriptions and Source Quality

The agent's system prompt now contains two additions:

- **`vertex_search` tool description** — tells the model to prefer this tool for
  questions answerable from internal documents, before reaching for `google_search`
- **`## Source Quality` section** — a five-tier source hierarchy that guides the
  agent to prefer peer-reviewed and government sources over blogs and aggregators

Let's inspect the relevant sections.

In [ ]:
from aieng.agent_evals.knowledge_qa.system_instructions import build_system_instructions

instructions = build_system_instructions()

# Extract and display the Tools section
tools_start = instructions.find("## Tools")
search_strategy_start = instructions.find("## Search Strategy")
tools_section = instructions[tools_start:search_strategy_start].strip()
console.print(Panel(tools_section, title="## Tools (from system prompt)", border_style="cyan"))

# Extract and display the Source Quality section
sq_start = instructions.find("## Source Quality")
verification_start = instructions.find("## CRITICAL")
sq_section = instructions[sq_start:verification_start].strip()
console.print(Panel(sq_section, title="## Source Quality (from system prompt)", border_style="yellow"))

## 3. Knowledge Base Question

We ask the agent a question that can only be answered from the private Vertex AI Search
data store. The agent's plan should call `vertex_search` first, and the answer should
reflect the grounded document content.

> **Skipping this section?** If `VERTEX_AI_DATASTORE_ID` is not set, the agent will
> fall back to `google_search` and likely return no result (the Northstar Analytics
> data does not exist on the public web by design).

In [ ]:
KB_QUESTION = "What is the monthly price for the Professional tier of Northstar Analytics, and what API rate limits does it include?"

if not config.vertex_datastore_id:
    console.print("[yellow]Skipping — VERTEX_AI_DATASTORE_ID not set.[/yellow]")
    kb_response = None
else:
    console.print(Panel(KB_QUESTION, title="KB Question", border_style="blue"))
    kb_agent = KnowledgeGroundedAgent(enable_planning=True)
    kb_response = await run_with_display(kb_agent, KB_QUESTION)
    display_response(
        console,
        kb_response.text,
        title="KB Answer",
        subtitle=f"Duration: {kb_response.total_duration_ms / 1000:.1f}s  |  Tools: {len(kb_response.tool_calls)}",
    )

In [ ]:
if kb_response is not None:
    t = Table(title="KB Question — Tool Call Sequence")
    t.add_column("#", style="dim", justify="right", width=3)
    t.add_column("Tool", style="cyan")
    t.add_column("Key Argument", style="white")
    for i, tc in enumerate(kb_response.tool_calls, 1):
        name = tc.get("name", "?")
        args = tc.get("args", {})
        key_arg = args.get("query", args.get("url", args.get("path", str(args))))[:80]
        style = "bold yellow" if name == "vertex_search" else ""
        t.add_row(str(i), f"[{style}]{name}[/{style}]" if style else name, key_arg)
    console.print(t)

    vertex_calls = [tc for tc in kb_response.tool_calls if tc.get("name") == "vertex_search"]
    if vertex_calls:
        console.print(f"[green]✓[/green] Agent called vertex_search {len(vertex_calls)} time(s) — grounded in private KB")
    else:
        console.print("[red]✗[/red] vertex_search was NOT called — check that VERTEX_AI_DATASTORE_ID is set")

## 4. Public Web Question

Now we ask a question that requires searching the public web. The agent should
use `google_search` followed by `web_fetch`, and should NOT call `vertex_search`
(the KB does not contain this information).

In [ ]:
WEB_QUESTION = "What was Canada's GDP growth rate in 2023 according to Statistics Canada?"

console.print(Panel(WEB_QUESTION, title="Web Question", border_style="blue"))
web_agent = KnowledgeGroundedAgent(enable_planning=True)
web_response = await run_with_display(web_agent, WEB_QUESTION)
display_response(
    console,
    web_response.text,
    title="Web Answer",
    subtitle=f"Duration: {web_response.total_duration_ms / 1000:.1f}s  |  Tools: {len(web_response.tool_calls)}",
)

In [ ]:
t = Table(title="Web Question — Tool Call Sequence")
t.add_column("#", style="dim", justify="right", width=3)
t.add_column("Tool", style="cyan")
t.add_column("Key Argument", style="white")
for i, tc in enumerate(web_response.tool_calls, 1):
    name = tc.get("name", "?")
    args = tc.get("args", {})
    key_arg = args.get("query", args.get("url", args.get("path", str(args))))[:80]
    t.add_row(str(i), name, key_arg)
console.print(t)

has_search = any(tc.get("name") == "google_search" for tc in web_response.tool_calls)
has_fetch = any(tc.get("name") == "web_fetch" for tc in web_response.tool_calls)
has_vertex = any(tc.get("name") == "vertex_search" for tc in web_response.tool_calls)

console.print()
console.print(f"google_search called: {'[green]yes[/green]' if has_search else '[red]no[/red]'}")
console.print(f"web_fetch called:     {'[green]yes[/green]' if has_fetch else '[red]no[/red]'}")
console.print(f"vertex_search called: {'[yellow]yes (unexpected)[/yellow]' if has_vertex else '[green]no (correct)[/green]'}")

## 5. Reading the Research Plan

With `enable_planning=True`, the agent produces a `ResearchPlan` before taking
any tool actions. The plan describes what the agent intends to do in each step.

We can inspect the plan to understand *why* the agent chose each tool — a key
input to the offline Plan Quality evaluator in Notebook 06.

In [ ]:
def display_plan(response, title: str) -> None:
    if not response or not response.plan or not response.plan.steps:
        console.print(f"[dim]{title}: no plan available[/dim]")
        return
    t = Table(title=title)
    t.add_column("Step", style="dim", justify="right", width=4)
    t.add_column("Description", style="white", min_width=50)
    t.add_column("Status", style="cyan", width=12)
    for step in response.plan.steps:
        status = str(step.status.value) if step.status else "pending"
        t.add_row(str(step.step_number), step.description, status)
    console.print(t)


if kb_response is not None:
    display_plan(kb_response, "Research Plan — KB Question")
    console.print()

display_plan(web_response, "Research Plan — Web Question")

## 6. Side-by-Side Comparison

A summary comparing the two execution profiles — highlights the key difference
between knowledge-base-grounded answers and public-web answers.

In [ ]:
def tool_summary(response) -> str:
    if not response:
        return "N/A"
    from collections import Counter
    counts = Counter(tc.get("name") for tc in response.tool_calls)
    return ", ".join(f"{name}×{n}" for name, n in counts.most_common())


cmp = Table(title="Execution Profile Comparison")
cmp.add_column("Metric", style="cyan", width=22)
cmp.add_column("KB Question", style="yellow", width=35)
cmp.add_column("Web Question", style="blue", width=35)

cmp.add_row("Question type", "Private KB (Northstar)", "Public web (Statistics Canada)")
cmp.add_row(
    "Primary tool",
    "vertex_search" if kb_response and any(tc.get("name") == "vertex_search" for tc in kb_response.tool_calls) else "google_search (fallback)",
    "google_search + web_fetch",
)
cmp.add_row(
    "Total tool calls",
    str(len(kb_response.tool_calls)) if kb_response else "N/A",
    str(len(web_response.tool_calls)),
)
cmp.add_row("Tool sequence", tool_summary(kb_response), tool_summary(web_response))
cmp.add_row(
    "Duration",
    f"{kb_response.total_duration_ms / 1000:.1f}s" if kb_response else "N/A",
    f"{web_response.total_duration_ms / 1000:.1f}s",
)
cmp.add_row(
    "Replan count",
    str(kb_response.replan_count) if kb_response else "N/A",
    str(web_response.replan_count),
)
cmp.add_row(
    "Sources cited",
    str(len(kb_response.sources)) + " (from KB)" if kb_response else "N/A",
    str(len(web_response.sources)) + " (from web)",
)
console.print(cmp)

## 7. Sources and Grounding

`AgentResponse.sources` contains the `GroundingChunk` objects resolved at the end
of `answer_async()`. For KB questions these are document URIs from the data store;
for web questions they are public URLs.

The offline **Source Validation** evaluator (Notebook 06) reads these to assess
authority and relevance.

In [ ]:
def display_sources(response, title: str) -> None:
    if not response or not response.sources:
        console.print(f"[dim]{title}: no sources[/dim]")
        return
    t = Table(title=title)
    t.add_column("#", style="dim", justify="right", width=3)
    t.add_column("Title", style="cyan", width=32)
    t.add_column("URI", style="dim")
    for i, src in enumerate(response.sources, 1):
        t.add_row(str(i), src.title[:30] if src.title else "(untitled)", src.uri[:70] if src.uri else "")
    console.print(t)


if kb_response is not None:
    display_sources(kb_response, "Sources — KB Answer")
    console.print()

display_sources(web_response, "Sources — Web Answer")

## Summary

In this notebook you:

1. **Inspected** which tools the agent has — `vertex_search` appears only when
   `VERTEX_AI_DATASTORE_ID` is set in the environment
2. **Reviewed** the system prompt additions — the `vertex_search` tool description
   and the five-tier Source Quality hierarchy
3. **Ran a KB question** and verified the agent called `vertex_search` with a
   well-formed query, returning a grounded answer from the private data store
4. **Ran a web question** and verified the agent used `google_search` + `web_fetch`,
   correctly avoiding `vertex_search` for public-web content
5. **Inspected the research plan** to understand how the agent planned each approach
6. **Compared execution profiles** — KB answers are faster and use one tool;
   web answers chain search → fetch → verify
7. **Inspected sources** — the `AgentResponse.sources` field the offline evaluator reads

### What's Next

- **Notebook 06** — run the offline evaluation suite on agent responses at dataset scale
- **Notebook 07** — wire online metrics (verification compliance, coherence, retry rate)
  into a production-style serving loop